# Лабораторная работа по теме: «Кластеризация»

### Подключение необходимых библиотек и загрузка датасета данных

In [31]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import plotly.express as px
import pandas as pd
import numpy as np

path = '/content/adult.csv'

df = pd.read_csv(path)
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Предобработка данных

In [34]:
columns_for_clustering = [
    'age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week',
    'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country'
]

df = df[columns_for_clustering + ['salary']]
df = df.replace(' ?', np.nan).dropna().reset_index(drop=True)

print(f"Размер данных после очистки: {df.shape}")

# Преобразование категориальных признаков в численные
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Выделяем матрицу признаков
X = df[columns_for_clustering].copy()

# Масштабирование данных
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Предобработка завершена.")

Размер данных после очистки: (32561, 14)
Предобработка завершена.


### Функции для кластеризации и визуализации

In [35]:
# Применяет KMeans и возвращает метки кластеров
def apply_KMeans(X_scaled, n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    return cluster_labels

# Применяет агломеративную кластеризацию и возвращает метки кластеров
def apply_agglomerative(X_scaled, n_clusters):
    agg = AgglomerativeClustering(n_clusters=n_clusters)
    cluster_labels = agg.fit_predict(X_scaled)
    return cluster_labels

# Вычисляет и выводит метрики качества кластеризации
def evaluate_clustering(X_scaled, labels, method_name):
    silhouette = silhouette_score(X_scaled, labels)
    calinski = calinski_harabasz_score(X_scaled, labels)
    print(f"{method_name}:")
    print(f"  Silhouette:          {silhouette:.4f}")
    print(f"  Calinski-Harabasz:   {calinski:.2f}")

### Функции для визуализации

In [36]:
def plot_pca_fast(X_scaled, cluster_labels, method_name):
    print(f"Создание визуализации для {method_name}...")

    # Применяем PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    df_viz = pd.DataFrame({
        'PCA1': X_pca[:, 0],
        'PCA2': X_pca[:, 1],
        'Cluster': cluster_labels.astype(str)
    })

    fig = px.scatter(
        df_viz,
        x='PCA1',
        y='PCA2',
        color='Cluster',
        title=f'PCA визуализация - {method_name}',
        labels={'Cluster': 'Кластер'},
        color_discrete_sequence=px.colors.qualitative.Set1,
        opacity=0.5
    )

    fig.update_traces(marker=dict(size=2))
    fig.update_layout(
        width=1000,
        height=600,
        plot_bgcolor='white'
    )

    fig.show()
    print(f"Визуализация для {method_name} завершена!\n")

# Формирование подвыборки
def plot_tsne_sample(X_scaled, cluster_labels, method_name, sample_size=3000):
    print(f"Создание t-SNE визуализации для {method_name} (подвыборка {sample_size} точек)...")

    np.random.seed(42)
    indices = np.random.choice(X_scaled.shape[0], size=sample_size, replace=False)
    X_sample = X_scaled[indices]
    labels_sample = cluster_labels[indices]

    tsne = TSNE(
        n_components=2,
        perplexity=30,
        random_state=42,
        n_iter=500,
        learning_rate='auto'
    )

    X_tsne = tsne.fit_transform(X_sample)

    df_viz = pd.DataFrame({
        't-SNE1': X_tsne[:, 0],
        't-SNE2': X_tsne[:, 1],
        'Cluster': labels_sample.astype(str)
    })

    fig = px.scatter(
        df_viz,
        x='t-SNE1',
        y='t-SNE2',
        color='Cluster',
        title=f't-SNE визуализация (подвыборка) - {method_name}',
        labels={'Cluster': 'Кластер'},
        color_discrete_sequence=px.colors.qualitative.Set1,
        opacity=0.6
    )

    fig.update_traces(marker=dict(size=3))
    fig.update_layout(
        width=1000,
        height=600,
        plot_bgcolor='white'
    )

    fig.show()
    print(f"Визуализация для {method_name} завершена\n")

### Кластеризация данных

In [37]:
# Применяем оба алгоритма для 2 и 5 кластеров
print("Выполняется кластеризация...")
KMeans_2 = apply_KMeans(X_scaled, 2)
KMeans_5 = apply_KMeans(X_scaled, 5)

Agglomerative_2 = apply_agglomerative(X_scaled, 2)
Agglomerative_5 = apply_agglomerative(X_scaled, 5)

print("Кластеризация выполнена успешно")

Выполняется кластеризация...
Кластеризация выполнена успешно


### Создание DataFrame с результатами

In [38]:
df_result = df.copy()
df_result['KMeans_2'] = KMeans_2
df_result['KMeans_5'] = KMeans_5
df_result['Agglomerative_2'] = Agglomerative_2
df_result['Agglomerative_5'] = Agglomerative_5

print("DataFrame с результатами кластеризации создан")

DataFrame с результатами кластеризации создан


### Визуализация результатов

In [39]:
# Используем PCA для визуализации
plot_pca_fast(X_scaled, KMeans_2, "KMeans (2 кластера)")
plot_pca_fast(X_scaled, KMeans_5, "KMeans (5 кластеров)")
plot_pca_fast(X_scaled, Agglomerative_2, "Agglomerative (2 кластера)")
plot_pca_fast(X_scaled, Agglomerative_5, "Agglomerative (5 кластеров)")

Создание визуализации для KMeans (2 кластера)...


Визуализация для KMeans (2 кластера) завершена!

Создание визуализации для KMeans (5 кластеров)...


Визуализация для KMeans (5 кластеров) завершена!

Создание визуализации для Agglomerative (2 кластера)...


Визуализация для Agglomerative (2 кластера) завершена!

Создание визуализации для Agglomerative (5 кластеров)...


Визуализация для Agglomerative (5 кластеров) завершена!



### Оценка качества кластеризации

In [40]:
evaluate_clustering(X_scaled, KMeans_2, "KMeans (2 кластера)")
evaluate_clustering(X_scaled, KMeans_5, "KMeans (5 кластеров)")
evaluate_clustering(X_scaled, Agglomerative_2, "Agglomerative (2 кластера)")
evaluate_clustering(X_scaled, Agglomerative_5, "Agglomerative (5 кластеров)")

KMeans (2 кластера):
  Silhouette:          0.1494
  Calinski-Harabasz:   4595.93
KMeans (5 кластеров):
  Silhouette:          0.1688
  Calinski-Harabasz:   3681.01
Agglomerative (2 кластера):
  Silhouette:          0.3067
  Calinski-Harabasz:   2676.05
Agglomerative (5 кластеров):
  Silhouette:          0.0998
  Calinski-Harabasz:   3154.83


### Анализ метрик качества

Лучший силуэт (0.3067) у Agglomerative с 2 кластерами — это говорит о том, что объекты в кластерах достаточно хорошо отделены друг от друга. Значение 0.3067 для социально-демографических данных считается хорошим результатом.

Лучшая компактность (4595.93) у KMeans с 2 кластерами — кластеры получились очень плотными, хотя границы между ними менее четкие (силуэт ниже).

Оптимальное число кластеров — 2, потому что:

При переходе к 5 кластерам у Agglomerative силуэт падает в 3 раза (с 0.3067 до 0.0998)

KMeans при 5 кластерах теряет в компактности (3681 против 4595)

Интересное наблюдение: KMeans при 5 кластерах показал чуть лучший силуэт (0.1688), чем при 2 (0.1494), но это всё равно остается в зоне "слабой структуры", а компактность при этом значительно упала.

Таким образом, для данного датасета наиболее информативным является разбиение на 2 кластера, при этом:

KMeans дает более компактные группы

Agglomerative обеспечивает лучшее разделение между группами


### Интерпретация кластеров

In [41]:
print("\n1. Распределение salary по кластерам:")
print("-" * 40)
print("KMeans (2 кластера):")
print(pd.crosstab(df_result['KMeans_2'], df_result['salary'], margins=True), '\n')
print("Agglomerative (2 кластера):")
print(pd.crosstab(df_result['Agglomerative_2'], df_result['salary'], margins=True), '\n')

print("\n2. Средние значения признаков по кластерам:")
print("-" * 40)
print("KMeans (2 кластера):")
print(df_result.groupby('KMeans_2')[columns_for_clustering].mean().round(2), '\n')
print("Agglomerative (2 кластера):")
print(df_result.groupby('Agglomerative_2')[columns_for_clustering].mean().round(2), '\n')


1. Распределение salary по кластерам:
----------------------------------------
KMeans (2 кластера):
salary    <=50K  >50K    All
KMeans_2                    
0         12741  1122  13863
1         11979  6719  18698
All       24720  7841  32561 

Agglomerative (2 кластера):
salary           <=50K  >50K    All
Agglomerative_2                    
0                23999  7070  31069
1                  721   771   1492
All              24720  7841  32561 


2. Средние значения признаков по кластерам:
----------------------------------------
KMeans (2 кластера):
            age  education-num  capital-gain  capital-loss  hours-per-week  \
KMeans_2                                                                     
0         32.83           9.70        401.53         48.91           35.27   
1         42.84          10.37       1578.93        115.77           44.27   

          workclass  education  marital-status  occupation  relationship  \
KMeans_2                                      

### Результаты интерпретации

**KMeans** (2 кластера):
- Кластер 0 (13863 чел.) — молодые (33 года), мало работают (35 ч/нед), низкий доход (>50K лишь у 8%)
- Кластер 1 (18698 чел.) — взрослые (43 года), много работают (44 ч/нед), доход выше (>50K у 36%)

**Agglomerative** (2 кластера):
- Кластер 0 (31069 чел.) — основная масса (95%), доход >50K у 23%
- Кластер 1 (1492 чел.) — элитная группа (5%), где 52% имеют доход >50K.  
  Характеристики: возраст 42 года, образование 11 лет, работа 43 ч/нед, высокие капитальные потери (инвестиции).

**Итог:** Оба алгоритма подтверждают, что высокий доход связан с возрастом, образованием, интенсивностью труда и инвестиционной активностью. Agglomerative наиболее точно выделил целевую группу.

### Вывод

В ходе выполнения работы был проведен анализ данных с использованием методов кластеризации KMeans и Agglomerative Clustering. Кластерный анализ позволил автоматически выделить социально-демографический портрет человека с высоким доходом: это люди старшего возраста, с высоким уровнем образования, высокой трудовой нагрузкой и наличием инвестиций. Полученные результаты хорошо согласуются с реальными экономическими закономерностями и подтверждают эффективность примененных методов кластеризации.